In [ ]:
import pandas as pd
df=pd.read_csv('retail_price.csv')
print(df.shape)

In [ ]:
df.describe(include='all')

## Data Cleaning

In [ ]:
# Missing value imputation for categorical and continuous variables
def missing_imputation(x, stats = 'mean'):
    if (x.dtypes == 'float64') | (x.dtypes == 'int64'):
        x = x.fillna(x.mean()) if stats == 'mean' else x.fillna(x.median())
    else:
        x = x.fillna(x.mode())
    return x

# An utility function to create dummy variable
def create_dummies(df, colname):
    col_dummies = pd.get_dummies(df[colname], prefix = colname, drop_first = True)
    df = pd.concat([df, col_dummies], axis = 1)
    df.drop(colname, axis = 1, inplace = True )
    return df

In [ ]:
df_conti_vars = df.loc[:, (df.dtypes == 'float64') | (df.dtypes == 'int64')]
df_cat_vars = df.loc[:, (df.dtypes == 'object')]

In [ ]:
df_cat_vars

## Outlier Treatment

In [ ]:
df_conti_vars = df_conti_vars.apply(lambda x: x.clip(lower = x.quantile(0.01), upper = x.quantile(0.99)))


## Missing Value Treatment

In [ ]:
df_conti_vars = df_conti_vars.apply(missing_imputation)
df_cat_vars = df_cat_vars.apply(missing_imputation)

In [ ]:
df_cat_vars = df[['product_id', 'product_category_name']]

for c_feature in ['product_id', 'product_category_name']:
    df_cat_vars[c_feature] = df_cat_vars[c_feature].astype('category')
    df_cat_vars = create_dummies(df_cat_vars, c_feature)

In [ ]:
df_new = pd.concat([df_conti_vars, df_cat_vars], axis = 1)
df_new

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.distplot(df_new.lag_price)
plt.show()

In [ ]:
# apply log transformation: log is rescalling the data and making the distribution normal
import numpy as np
df_new['lag_price'] = np.log(df_new['lag_price']+1)

# Distribution of variables
sns.distplot(df_new.lag_price)
plt.show()

## Scaling of dataset

In [ ]:
from sklearn.preprocessing import StandardScaler
TargetVariable=['lag_price']
 
X=df_new.drop(columns=['lag_price'])
y=df_new[TargetVariable].values
 
### Sandardization of data ###
PredictorScaler=StandardScaler()
TargetVarScaler=StandardScaler()
 
# Storing the fit object for later reference
PredictorScalerFit=PredictorScaler.fit(X)
TargetVarScalerFit=TargetVarScaler.fit(y)
 
# Generating the standardized values of X and y
X=PredictorScalerFit.transform(X)
y=TargetVarScalerFit.transform(y)


## Testing and training

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
 
# Quick sanity check with the shapes of Training and testing datasets
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
# importing the libraries
from keras.models import Sequential
from keras.layers import Dense
 
# create ANN model
model = Sequential()
 
# Defining the Input layer 
model.add(Dense(units=4, input_dim=85, kernel_initializer='normal', activation='relu'))
 
# Defining the Second(Hidden) layer of the model
# after the first layer we don't have to specify input_dim as keras configure it automatically
#model.add(Dense(units=3, kernel_initializer='normal', activation='tanh'))
 
# The output neuron is a single fully connected node 
# Since we will be predicting a single number
model.add(Dense(1, kernel_initializer='normal'))
 
# Compiling the model
model.compile(loss='mean_squared_error', optimizer='adam')
 
# Fitting the ANN to the Training set
model.fit(X_train, y_train ,batch_size = 20, epochs = 45, verbose=1)

In [ ]:
predictors=pd.DataFrame(data=df_new,columns=df_new.drop(columns='lag_price').columns)

In [ ]:
# Generating Predictions on testing data
Predictions=model.predict(X_test)

# Scaling the predicted Price data back to original price scale
Predictions=TargetVarScalerFit.inverse_transform(Predictions)
 
# Scaling the y_test Price data back to original price scale
y_test_orig=TargetVarScalerFit.inverse_transform(y_test)
 
# Scaling the test data back to original scale
Test_Data=PredictorScalerFit.inverse_transform(X_test)
 
TestingData=pd.DataFrame(data=Test_Data, columns=predictors.columns)
TestingData['Value']=y_test_orig
TestingData['PredictedValue']=Predictions
TestingData.head(10)

In [ ]:
#print(TestingData.head(10))
print(TestingData.iloc[0:10,0:12])
print(TestingData.iloc[0:10,85:87])


In [ ]:
TestingData.columns

In [ ]:
result=pd.DataFrame({'Quantity':TestingData['qty'],'freight_price':TestingData['freight_price'],'unit_price':TestingData['unit_price'],'product_name_length':TestingData['product_name_lenght'],
                    'product_description_length':TestingData['product_description_lenght'],'product_photos_qty':TestingData['product_photos_qty'],'product_weight_g':TestingData['product_weight_g'],
                    'product_score':TestingData['product_score'],'customers':TestingData['customers'],'scaled_value':TestingData['Value'],
                    'predicted_scaled_value':TestingData['PredictedValue']})

result.head(10)

In [ ]:
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
print('Accuracy of the model is',r2_score(y_test_orig,Predictions))
print('MSE',mean_squared_error(y_test_orig,Predictions))
print('MAE',mean_absolute_error(y_test_orig,Predictions))

## final result

In [ ]:
fresult=pd.DataFrame({'Actual':result['scaled_value'],'Predicted':result['predicted_scaled_value']})
fresult['Actual']=np.exp(fresult['Actual'])
fresult['Predicted']=np.exp(fresult['Predicted'])
fresult=fresult.astype({'Actual':int,'Predicted':int})

fresult

In [ ]:
f = plt.figure()
f.set_figwidth(8)
f.set_figheight(8)
plt.rcParams['axes.grid'] = True
sns.lineplot(x=TestingData['Value'],y=TestingData['PredictedValue'])